In [1]:
# dementia_qwen_predict_fewshot_noreasoning.py
import os
import re
import sys
from pathlib import Path
import torch
import pandas as pd
import csv
from typing import List, Dict, Optional
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

# Optional: TF-IDF retrieval for few-shot
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# ----------------- CONFIG -----------------
DATA_DIR = r"E:\Thesis\cookie_control_dementia"
MODEL_NAME = "Qwen/Qwen3-4B"
HF_TOKEN = os.environ.get("HF_TOKEN")
OUTPUT_CSV = r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass.csv"

USE_FEWSHOT = True
FEWSHOT_K = 3
EXAMPLES_CSV: Optional[str] = None  # if you have labeled examples CSV

GEN_MAX_NEW_TOKENS = 32768
GEN_TEMPERATURE = 0.3
GEN_TOP_P = 0.9

# ================= BUILTIN FEW-SHOT EXAMPLES (from actual dataset) =================
BUILTIN_EXAMPLES: List[Dict] = [
    {
        "filename": "002-0.txt",
        "transcript": """the scene is <in the> [/] in the kitchen . 
the mother is wiping dishes and the water is running on the floor . 
<a child is tryin(g) to get> [//] a boy is tryin(g) to get cookies outta [: out of] a jar and he's about to tip over on a stool . 
&-uh the little girl is reacting to his falling . 
&-uh it seems to be summer out . 
the window is open . 
the curtains are blowing . 
it must be a gentle breeze . 
there's grass outside in the garden . 
&-uh mother's finished certain of the [/] the dishes . 
kitchen's very tidy . [+ gram] 
the mother seems to have nothing in the house to eat except cookies in the cookie jar . 
&-uh the children look to be almost about the same size . 
perhaps they're twins . 
they're dressed for summer warm weather . 
&-um you want more ? [+ exc] 
+< the mother's in a short sleeve dress . 
+< I'll hafta say it's warm . [+ exc]
""",
        "label": "Control",
        "mmse": 30
    },
    {
        "filename": "001-0.txt",
        "transcript": """mhm . [+ exc] 
+< alright . [+ exc] 
there's &-um a young boy that's getting a cookie jar . 
and it [//] he's &-uh in bad shape because &-uh the thing is fallin(g) over . 
and in the picture the mother is washin(g) dishes and doesn't see it . 
and so <is the> [//] the water is overflowing in the sink . 
and the dishes might <get falled [* +ed] over if you don't> [//] fell [//] fall over there [/] there if you don't get it . 
and it [//] there [//] it's a picture of a kitchen window . 
and the curtains are very &-uh distinct . 
but the water is &+flow still flowing .""",
        "label": "ProbableAD",
        "mmse": 18
    },
    {
        "filename": "034-1.txt",
        "transcript": """boy &-uh taking cookies out of a cookie jar . [+ gram] 
the stool is falling . 
the little girl is reaching . 
water is running out o(f) the faucet . 
the water is overflowing the sink . 
woman is <washin(g) dishes> [//] (...) drying dishes . [+ gram] 
there's nothing to indicate xxx and I don't see any more action . [+ exc] 

""",
        "label": "Control",
        "mmse": 28
    },
    {        
        "filename": "216-1.txt",
        "transcript": """now honey I &+w &+l &+l &+ha &+w had it was in the kitchen and I was the oldest ten . [+ gram] [+ exc]
and if we made a mess like that you'd get a kick in the ass . [+ exc]
+< well ‡ we have &-uh spillin(g) of the water .
and a kid with his cookie jar . [+ gram]
and a stool is turned over .
and a mother's runnin(g) the water on the floor .
and what else do you want from that ? [+ exc]
it &+s looks like somebody's layin(g) out in the grass doesn't it ?
+< and a kid in the cookie jar . [+ gram]
and a tilted &+s stool . [+ gram]
what more do you want ? [+ exc]
+< the [/] &+wa the water rollin(g) on the floor . [+ gram]""",
        "label": "ProbableAD",
        "mmse": 16
    },
    {
        "filename": "684-0.txt",
        "transcript": """&-uh the boy is reaching into the cookie jar .
he's falling off the stool .
the little girl is reaching for a cookie .
mother is drying the dishes .
the sink is running over .
mother's getting her feet wet .
they all have shoes on .
there's <a cup> [//] two cups and a saucer on the sink .
the window has draw [//] withdrawn [: drawn] [* s:r] drapes .
you look out on the driveway .
there's kitchen cabinets .
oh what's happening . [+ exc]
mother is looking out the window .
the girl is touching her lips .
the boy is standing on his right foot .
his left foot is sort_of up in the air .
mother's right foot is flat on the floor and <her left> [//] she's on her left toe .
&-uh she's holding the dish cloth in her right hand and the plate she is drying in her left .
I think I've run out of +/. [+ exc]
+< yeah &=laughs . [+ exc]
""",
        "label": "Control",
        "mmse": 30
    },
    {
        "filename": "711-0.txt",
        "transcript": """oh ‡ you want me to tell you . [+ exc]
the mother and her two children . [+ gram]
and the children are getting in the cookie jar .
and she's doing the dishes and spilling the water .
and she had the spigot on .
and she didn't know it perhaps .
&=coughs pardon me . [+ exc]
and they're looking out into the garden from the kitchen window .
it's open .
and the &-uh cookies must be pretty good they're eating .
<the tair [: chair] [* p:w-ret]> [//] &-uh the chair [: stool] [* s:r] is &+t &-uh tilting and he's gonna fall off .
and &-uh <the lady> [//] the mother's splashing her shoes and stockings all up overflowing the water .
and there's &-um &-uh a window and curtains on the window .
and I can see some trees outside there .
and [/] and there's dishes that &+h had been washed .
and she's dryin(g) them .
and there's some shrub out there and +...
""",
        "label": "ProbableAD",
        "mmse": 25
    }
]

# ----------------- HF SANITY CHECK -----------------
def hf_sanity_check(model_name: str, token: Optional[str] = None) -> bool:
    api = HfApi()
    try:
        info = api.model_info(model_name, token=token)
        print(f"HF CHECK OK: {info.modelId} (private={info.private})")
        return True
    except Exception as e:
        print(f"HF CHECK ERROR: {type(e).__name__}: {e}", file=sys.stderr)
        return False

# ----------------- TOKENIZER / MODEL LOADERS -----------------
def load_qwen_tokenizer(model_name: str, hf_token: Optional[str] = None):
    kwargs = {"trust_remote_code": True}
    if hf_token:
        kwargs["token"] = hf_token
    try:
        tok = AutoTokenizer.from_pretrained(model_name, **kwargs)
    except Exception as e_remote:
        print(f"[Tokenizer] remote load failed: {type(e_remote).__name__}: {e_remote}", file=sys.stderr)
        local_dir = snapshot_download(repo_id=model_name, token=hf_token)
        tok = AutoTokenizer.from_pretrained(local_dir, trust_remote_code=True, local_files_only=True)
    return tok

def load_qwen_model(model_name: str, hf_token: Optional[str] = None):
    kwargs = {
        "trust_remote_code": True,
        "torch_dtype": "auto",
        "device_map": "auto"
    }
    if hf_token:
        kwargs["token"] = hf_token
    try:
        return AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    except Exception as e_remote:
        print(f"[Model] remote load failed: {type(e_remote).__name__}: {e_remote}", file=sys.stderr)
        local_dir = snapshot_download(repo_id=model_name, token=hf_token)
        return AutoModelForCausalLM.from_pretrained(
            local_dir, 
            trust_remote_code=True, 
            local_files_only=True, 
            torch_dtype="auto", 
            device_map="auto"
        )

# ----------------- PROMPT / PARSING -----------------
def extract_patient_info(filename: str):
    match = re.match(r'(\d{3})', filename)
    return match.group(1) if match else None

def create_prediction_prompt(filename: str, transcript: str) -> str:
    return f"""You are an expert neuropsychologist specializing in dementia assessment. Analyze the following Cookie Theft picture description transcript.

**File:** {filename}

**Transcript:**
{transcript}

**Task:**
Based on this transcript, predict:

1. **Diagnosis Label**: Choose ONE from:
   - ProbableAD
   - Control

2. **MMSE Score**: Predict a score between 0-30.

**Response Format (only this, no extra text):**
Label: [ProbableAD/Control]
MMSE: [0-30]
Confidence: [Low/Medium/High]"""

def parse_llm_response(response: str) -> Dict:
    label_match = re.search(r'Label:\s*(\w+)', response)
    mmse_match = re.search(r'MMSE:\s*(\d+)', response)
    conf_match = re.search(r'Confidence:\s*(\w+)', response)
    return {
        'label': label_match.group(1) if label_match else 'Unknown',
        'mmse': int(mmse_match.group(1)) if mmse_match else -1,
        'confidence': conf_match.group(1) if conf_match else 'Unknown',
        'raw': response
    }

# ----------------- FEWSHOT LOGIC -----------------
def load_examples_from_csv(csv_path: str) -> List[Dict]:
    examples = []
    with open(csv_path, newline='', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            transcript = (row.get("transcript") or row.get("text") or "").strip()
            label = (row.get("label") or "").strip()
            mmse = row.get("mmse") or ""
            examples.append({
                "filename": row.get("filename") or "example",
                "transcript": transcript,
                "label": label,
                "mmse": int(mmse) if str(mmse).isdigit() else None
            })
    return examples

def pick_k_most_similar_examples(query: str, examples: List[Dict], k: int) -> List[Dict]:
    if not examples:
        return []
    if SKLEARN_AVAILABLE and len(examples) > 1:
        docs = [ex["transcript"] for ex in examples]
        vect = TfidfVectorizer(max_features=4096, stop_words='english').fit(docs + [query])
        doc_mat = vect.transform(docs)
        q_vec = vect.transform([query])
        sims = cosine_similarity(q_vec, doc_mat)[0]
        idxs = sims.argsort()[::-1][:k]
        return [examples[i] for i in idxs]
    return examples[:k]

def build_fewshot_block(examples: List[Dict]) -> str:
    if not examples:
        return ""
    parts = []
    for ex in examples:
        parts.append(
            f"--- Example: {ex.get('filename','example')} ---\n"
            f"Transcript:\n{ex.get('transcript','')}\n\n"
            f"Label: {ex.get('label','')}\nMMSE: {ex.get('mmse','')}\n"
        )
    return "\n".join(parts)

# ----------------- CHAT RENDER / GENERATION (Qwen3 style) -----------------
def generate_from_model(messages: List[Dict], tokenizer, model) -> str:
    """Generate response using Qwen3 chat template."""
    
    # Apply chat template without thinking mode
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        do_sample=True
    )
    
    # Extract only the new tokens
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    
    return content

# ----------------- PREDICTION FUNCTIONS -----------------
def predict_visit_fewshot(filename: str, transcript: str, tokenizer, model,
                          use_fewshot: bool = USE_FEWSHOT,
                          k: int = FEWSHOT_K,
                          examples_csv: Optional[str] = EXAMPLES_CSV,
                          builtin_examples: Optional[List[Dict]] = BUILTIN_EXAMPLES) -> str:
    examples = []
    if use_fewshot:
        if examples_csv and Path(examples_csv).exists():
            examples = load_examples_from_csv(examples_csv)
        else:
            examples = builtin_examples
        chosen = pick_k_most_similar_examples(transcript, examples, k)
        fewshot_block = build_fewshot_block(chosen)
    else:
        fewshot_block = ""

    main_prompt = create_prediction_prompt(filename, transcript)
    if fewshot_block:
        combined = (
            "Here are some labeled examples of Cookie Theft descriptions.\n\n"
            f"{fewshot_block}\n\n"
            "Now analyze the new transcript and predict in the same format.\n\n"
            f"{main_prompt}"
        )
    else:
        combined = main_prompt

    messages = [
        {"role": "system", "content": "You are an expert neuropsychologist diagnosing dementia from speech."},
        {"role": "user", "content": combined}
    ]
    
    return generate_from_model(messages, tokenizer, model)

# ----------------- MAIN -----------------
def main():
    print("Checking model on HuggingFace...")
    hf_sanity_check(MODEL_NAME, HF_TOKEN)

    print("Loading tokenizer and model...")
    tokenizer = load_qwen_tokenizer(MODEL_NAME, hf_token=HF_TOKEN)
    model = load_qwen_model(MODEL_NAME, hf_token=HF_TOKEN)
    print("Model loaded.")

    files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".txt"))
    print(f"Found {len(files)} transcript files")

    results = []
    for i, filename in enumerate(files, 1):
        pid = extract_patient_info(filename)
        path = Path(DATA_DIR) / filename
        print(f"\n[{i}/{len(files)}] Processing {filename} (Patient ID: {pid})")
        try:
            with open(path, "r", encoding="utf-8") as f:
                transcript = f.read().strip()
            if not transcript:
                raise ValueError("Empty transcript")
            
            response = predict_visit_fewshot(filename, transcript, tokenizer, model)
            
            # Parse the response
            prediction = parse_llm_response(response)
            
            print(f"Label: {prediction['label']} | MMSE: {prediction['mmse']} | Conf: {prediction['confidence']}")
            
            # Remove .txt from filename for 'id'
            file_id = re.sub(r'\.txt$', '', filename)
            
            results.append({
                "id": file_id,
                "confidence": prediction["confidence"],
                "mmse": prediction["mmse"],
                "dx": prediction["label"]
            })
        except Exception as e:
            print(f"ERROR: {e}", file=sys.stderr)
            file_id = re.sub(r'\.txt$', '', filename)
            results.append({
                "id": file_id,
                "confidence": "N/A",
                "mmse": -1,
                "dx": "Error"
            })

    # Save id, confidence, mmse, dx
    df = pd.DataFrame(results, columns=["id", "confidence", "mmse", "dx"])
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Results saved to: {OUTPUT_CSV}")

    print("\nSample predictions:")
    print(df.head())

if __name__ == "__main__":
    main()

c:\pytorch-cuda-workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checking model on HuggingFace...
HF CHECK OK: Qwen/Qwen3-4B (private=False)
Loading tokenizer and model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.88s/it]


Model loaded.
Found 552 transcript files

[1/552] Processing 001-0.txt (Patient ID: 001)
Label: ProbableAD | MMSE: 22 | Conf: High

[2/552] Processing 001-2.txt (Patient ID: 001)
Label: ProbableAD | MMSE: 22 | Conf: Medium

[3/552] Processing 002-0.txt (Patient ID: 002)
Label: Control | MMSE: 28 | Conf: High

[4/552] Processing 002-1.txt (Patient ID: 002)
Label: Control | MMSE: 28 | Conf: High

[5/552] Processing 002-2.txt (Patient ID: 002)
Label: ProbableAD | MMSE: 22 | Conf: Medium

[6/552] Processing 002-3.txt (Patient ID: 002)
Label: Control | MMSE: 28 | Conf: High

[7/552] Processing 003-0.txt (Patient ID: 003)
Label: ProbableAD | MMSE: 20 | Conf: Medium

[8/552] Processing 005-0.txt (Patient ID: 005)
Label: ProbableAD | MMSE: 19 | Conf: Medium

[9/552] Processing 005-2.txt (Patient ID: 005)
Label: Control | MMSE: 26 | Conf: Medium

[10/552] Processing 006-2.txt (Patient ID: 006)
Label: ProbableAD | MMSE: 24 | Conf: Medium

[11/552] Processing 006-3.txt (Patient ID: 006)
Label: Co

In [3]:
# eval_fewshot_vs_actual.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# ---------- Paths (as you specified) ----------
actual_path = r"E:\Thesis\2_classes.csv"
pred_path   = r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass.csv"

# ---------- Load CSVs ----------
print("Loading CSVs...")
df_true = pd.read_csv(actual_path)
df_pred = pd.read_csv(pred_path)
print("Loaded actual:", actual_path)
print("Loaded predictions:", pred_path)

# ---------- Column mapping ----------
# Actual CSV columns: id, mmse, dx (or similar)
# Prediction CSV columns: id, confidence, mmse, dx, reasoning

# For actual data - use mmse and dx columns
mmse_true_col = "mmse"
label_true_col = "dx"

# For prediction data - use mmse and dx columns
mmse_pred_col = "mmse"
label_pred_col = "dx"

print(f"Using (actual) mmse column: '{mmse_true_col}'  label column: '{label_true_col}'")
print(f"Using (pred)   mmse column: '{mmse_pred_col}'  label column: '{label_pred_col}'")

# ---------- Rename for consistency ----------
df_true = df_true.rename(columns={mmse_true_col: "actual_mmse", label_true_col: "actual_label"})
df_pred = df_pred.rename(columns={mmse_pred_col: "pred_mmse", label_pred_col: "pred_label"})

# ---------- Ensure 'id' exists in both ----------
if "id" not in df_true.columns or "id" not in df_pred.columns:
    raise KeyError("Both CSVs must contain an 'id' column for matching (patient id).")

# ---------- Merge on id ----------
df_merged = pd.merge(df_pred, df_true, on="id", how="inner", suffixes=("_pred", "_true"))
if df_merged.empty:
    raise ValueError("Merge resulted in 0 rows. Check that 'id' values match between files.")

print(f"Merged {len(df_merged)} rows on 'id'.")

# ---------- Normalize label strings ----------
df_merged["pred_label_norm"] = df_merged["pred_label"].astype(str).str.strip().str.lower()
df_merged["actual_label_norm"] = df_merged["actual_label"].astype(str).str.strip().str.lower()
df_merged["label_match"] = df_merged["pred_label_norm"] == df_merged["actual_label_norm"]

# ---------- Numeric MMSE ----------
df_merged["pred_mmse"] = pd.to_numeric(df_merged["pred_mmse"], errors="coerce")
df_merged["actual_mmse"] = pd.to_numeric(df_merged["actual_mmse"], errors="coerce")
df_merged["mmse_diff"] = (df_merged["pred_mmse"] - df_merged["actual_mmse"]).abs()

# ---------- Metrics ----------
label_accuracy = df_merged["label_match"].mean() * 100
mae = df_merged["mmse_diff"].mean()
corr = df_merged["pred_mmse"].corr(df_merged["actual_mmse"])

# Optional: per-class precision/recall/f1
labels = sorted(set(df_merged["actual_label_norm"].unique()) | set(df_merged["pred_label_norm"].unique()))
y_true = df_merged["actual_label_norm"]
y_pred = df_merged["pred_label_norm"]
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)

prf_df = pd.DataFrame({
    "label": labels,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
})

# ---------- Print summary ----------
print("\n📊 Evaluation Results")
print(f"Samples evaluated: {len(df_merged)}")
print(f"Label Accuracy: {label_accuracy:.2f}% ({int(df_merged['label_match'].sum())}/{len(df_merged)})")
print(f"MMSE Mean Absolute Error (MAE): {mae:.3f}")
print(f"MMSE Pearson correlation: {corr:.3f}\n")

print("Per-class precision/recall/f1:")
print(prf_df.to_string(index=False))

# ---------- Save merged CSV ----------
out_csv = r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_vs_actual_evaluation.csv"
df_merged.to_csv(out_csv, index=False)
print(f"\nSaved merged evaluation CSV: {out_csv}")

# ---------- Confusion matrix ----------
cm = confusion_matrix(df_merged["actual_label_norm"], df_merged["pred_label_norm"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (normalized labels)")
confusion_path = r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_confusion_matrix_labels.png"
plt.savefig(confusion_path, bbox_inches="tight")
plt.close()
print(f"Saved confusion matrix to: {confusion_path}")

# ---------- Scatter plot: actual vs predicted MMSE ----------
plt.figure(figsize=(6,6))
plt.scatter(df_merged["actual_mmse"], df_merged["pred_mmse"], alpha=0.7)
xmin = min(df_merged["actual_mmse"].min(), df_merged["pred_mmse"].min())
xmax = max(df_merged["actual_mmse"].max(), df_merged["pred_mmse"].max())
plt.plot([xmin, xmax], [xmin, xmax], linestyle="--", color="gray")
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title("Predicted vs Actual MMSE")
plt.grid(True)
scatter_path = r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_mmse_scatter.png"
plt.savefig(scatter_path, bbox_inches="tight")
plt.close()
print(f"Saved MMSE scatter plot to: {scatter_path}")

# ---------- Save per-class PRF ----------
prf_out = r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_per_class_prf.csv"
prf_df.to_csv(prf_out, index=False)
print(f"Saved per-class precision/recall/f1 to: {prf_out}")

# ---------- Small summary file ----------
summary = {
    "n_samples": [len(df_merged)],
    "label_accuracy_pct": [label_accuracy],
    "mmse_mae": [mae],
    "mmse_corr": [corr]
}
pd.DataFrame(summary).to_csv(r"E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_eval_summary.csv", index=False)
print("Saved eval summary to: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_eval_summary.csv")

Loading CSVs...
Loaded actual: E:\Thesis\2_classes.csv
Loaded predictions: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass.csv
Using (actual) mmse column: 'mmse'  label column: 'dx'
Using (pred)   mmse column: 'mmse'  label column: 'dx'
Merged 498 rows on 'id'.

📊 Evaluation Results
Samples evaluated: 498
Label Accuracy: 57.23% (285/498)
MMSE Mean Absolute Error (MAE): 5.494
MMSE Pearson correlation: 0.256

Per-class precision/recall/f1:
     label  precision   recall       f1  support
   control   0.631579 0.296296 0.403361      243
probablead   0.554688 0.835294 0.666667      255

Saved merged evaluation CSV: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_vs_actual_evaluation.csv
Saved confusion matrix to: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_confusion_matrix_labels.png
Saved MMSE scatter plot to: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_mmse_scatter.png
Saved per-class precision/recall/f1 to: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_per_

<>:136: SyntaxWarning: invalid escape sequence '\T'
<>:136: SyntaxWarning: invalid escape sequence '\T'
C:\Users\NOWRIN\AppData\Local\Temp\ipykernel_14652\1984884803.py:136: SyntaxWarning: invalid escape sequence '\T'
  print("Saved eval summary to: E:\Thesis\qwen3-4B_fewshot_multiexample_binaryclass_eval_summary.csv")
